# 03 — Ingeniería de Variables (Feature Engineering)

**Input:** `data/dataset_imputed.csv` (salida del notebook `02_imputation_pipeline.ipynb`)  
**Output:** `data/dataset_engineered.csv` — dataset listo para K-Means y modelos supervisados

---

## Decisiones de diseño

| Variable | Estrategia | Justificación |
|---|---|---|
| `admission.test` | Normalizar PAA/PAL → 0-1, imputar mediana | MCAR — el test no tomado no depende del perfil |
| `admission.rubric` | Imputar mediana | MCAR — ya imputado en pipeline v2, por si acaso |
| `total.life.activities` | → `has_extracurriculars` binaria | MAR — la ausencia indica no participación, no dato perdido |
| `physical/cultural/social` | → `has_physical`, `has_cultural`, `has_social` | MAR — granularidad adicional por tipo de actividad |
| `first.generation` | → `first_gen_enc` (-1/0/1) | MNAR — el -1 captura el patrón de ausencia |
| `max.degree.parents` | → `educ_padres_max` ordinal (-1 a 3) | Ordinal natural, -1 = sin información |
| `parents.exatec` | → `parents_exatec_enc` binaria | Binario sin información relevante en ausencia |
| `gender` | → `is_male` binaria | Binario directo |
| `tec.no.tec` | → `estuvo_prepa_tec` binaria | Binario directo |
| `foreign` | → one-hot (2 columnas) | Nominal sin orden |
| `zone.type` | → `zone_enc` ordinal + one-hot | Rural<Semiurban<Urban; NaN → 0 en ordinal |
| `socioeconomic.level` | → `socioec_enc` ordinal 0-7 | Escala ordinal natural |
| `social.lag` | → `social_lag_enc` ordinal 0-3 | Escala ordinal natural |
| `school` | → `school_enc` (label) | Alta cardinalidad — label encode para modelos de árbol |
| `region` | → `region_enc` (label) | Cardinalidad media — label encode |
| `total.scholarship.loan` | → `apoyo_financiero` | Renombre semántico |

**Columnas eliminadas (fuga o irrelevantes):**
`student.id`, `level`, `average.first.period`, `failed.subject.first.period`,
`dropped.subject.first.period`, `dropout.semester`, `scholarship.type`,
`scholarship.perc`, `loan.perc`, `program`, `id.school.origin`,
`school.cost`, `father/mother.education.*`, `educational.model` (redundante con `regime`)

## 0. Imports y rutas

In [6]:
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.preprocessing import LabelEncoder

DATA_DIR   = Path('../')
INPUT_CSV  = DATA_DIR / 'dataset_imputed.csv'
OUTPUT_CSV = DATA_DIR / 'dataset_engineered.csv'

assert INPUT_CSV.exists(), f"No se encontró {INPUT_CSV} — ejecutar 02_imputation_pipeline primero"

df_raw = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"✓ dataset_imputed.csv cargado: {df_raw.shape}")
print(f"  Columnas: {df_raw.columns.tolist()}")

✓ dataset_imputed.csv cargado: (77517, 58)
  Columnas: ['student.id', 'generation', 'educational.model', 'level', 'gender', 'age', 'max.degree.parents', 'father.education.complete', 'father.education.summary', 'mother.education.complete', 'mother.education.summary', 'parents.exatec', 'father.exatec', 'mother.exatec', 'tec.no.tec', 'foreign', 'zone.type', 'first.generation', 'school', 'program', 'region', 'PNA', 'admission.test', 'online.test', 'english.evaluation', 'admission.rubric', 'general.math.eval', 'retention', 'FTE', 'scholarship.perc', 'scholarship.type', 'loan.perc', 'total.scholarship.loan', 'school.cost', 'id.school.origin', 'socioeconomic.level', 'social.lag', 'average.first.period', 'failed.subject.first.period', 'dropped.subject.first.period', 'dropout.semester', 'physical.education', 'cultural.diffusion', 'student.society', 'total.life.activities', 'athletic.sports', 'art.culture', 'student.society.leadership', 'life.work.mentoring', 'wellness.activities', 'regime', 'to

## 1. Filtrar nivel universitario

In [7]:
# El pipeline de imputación ya puede incluir todos los niveles — filtramos aquí
if 'level' in df_raw.columns:
    df = df_raw[df_raw['level'] == 'Undergraduate'].copy().reset_index(drop=True)
    print(f"Filtro 'level == Undergraduate': {len(df):,} / {len(df_raw):,} registros")
else:
    df = df_raw.copy()
    print(f"Columna 'level' no presente — usando todos los registros: {len(df):,}")

print(f"Desertó (retention==0): {(df['retention']==0).sum():,} ({(df['retention']==0).mean()*100:.1f}%)")
print(f"Retuvo  (retention==1): {(df['retention']==1).sum():,} ({(df['retention']==1).mean()*100:.1f}%)")

Filtro 'level == Undergraduate': 77,517 / 77,517 registros
Desertó (retention==0): 6,813 (8.8%)
Retuvo  (retention==1): 70,704 (91.2%)


## 2. Eliminar columnas con fuga de datos e irrelevantes

In [8]:
DROP_COLS = [
    # Identificadores
    'student.id', 'id.school.origin',
    # Nivel constante (solo Undergraduate)
    'level',
    # FUGA: calificaciones y eventos del primer semestre (solo disponibles post-deserción)
    'average.first.period',
    'failed.subject.first.period',
    'dropped.subject.first.period',
    'dropout.semester',
    # Redundantes o de alta cardinalidad sin valor predictivo
    'program',         # demasiados valores únicos, no comparables entre regímenes
    # Financieras redundantes (usamos total.scholarship.loan)
    'scholarship.type',
    'scholarship.perc',
    'loan.perc',
    # Redundante con regime
    'educational.model',
    # school.cost: redundante con socioeconomic.level
    'school.cost',
    # Detalle de educación de padres (usamos max.degree.parents → educ_padres_max)
    'father.education.complete', 'father.education.summary',
    'mother.education.complete', 'mother.education.summary',
    # exatec por separado (usamos parents.exatec)
    'father.exatec', 'mother.exatec',
]

existing_drops = [c for c in DROP_COLS if c in df.columns]
df.drop(columns=existing_drops, inplace=True)
print(f"Columnas eliminadas ({len(existing_drops)}): {existing_drops}")
print(f"Shape tras drops: {df.shape}")

Columnas eliminadas (19): ['student.id', 'id.school.origin', 'level', 'average.first.period', 'failed.subject.first.period', 'dropped.subject.first.period', 'dropout.semester', 'program', 'scholarship.type', 'scholarship.perc', 'loan.perc', 'educational.model', 'school.cost', 'father.education.complete', 'father.education.summary', 'mother.education.complete', 'mother.education.summary', 'father.exatec', 'mother.exatec']
Shape tras drops: (77517, 39)


## 3. MCAR — Normalización de `admission.test`

PAA (escala 400–1600) y PAL (escala 0–100) se unifican en 0–1.  
Los NaN residuales se imputan con la mediana del régimen PreTec21 (train).

In [9]:
def norm_admission_test(val):
    """PAA (>100) → (val-400)/1200; PAL (0-100) → val/100; NaN → NaN."""
    if pd.isna(val): return np.nan
    try:
        s = str(val).strip()
        if s.lower().startswith('does not'): return np.nan
        f = float(s)
        if f > 100:   return max(0.0, (f - 400) / 1200.0)   # PAA
        elif f >= 0:  return f / 100.0                        # PAL
    except Exception:
        pass
    return np.nan

df['admission_test_norm'] = df['admission.test'].apply(norm_admission_test)
df.drop(columns=['admission.test'], inplace=True)

# Imputar mediana de PreTec21 (no del total para evitar data leakage)
pretec21_mask = df['regime'] == 'PreTec21'
med_at = df.loc[pretec21_mask, 'admission_test_norm'].median()
n_nan  = df['admission_test_norm'].isna().sum()
df['admission_test_norm'].fillna(med_at, inplace=True)
print(f"admission_test_norm: {n_nan:,} NaN imputados con mediana PreTec21 = {med_at:.3f}")
print(f"  min={df['admission_test_norm'].min():.3f}  max={df['admission_test_norm'].max():.3f}  "
      f"mean={df['admission_test_norm'].mean():.3f}")

admission_test_norm: 0 NaN imputados con mediana PreTec21 = 0.804
  min=0.010  max=1.000  mean=0.796


## 4. MCAR — Imputación residual de `admission.rubric`

In [10]:
# El pipeline MICE ya imputó admission.rubric, pero por si acaso quedan NaN
n_nan_rub = df['admission.rubric'].isna().sum()
if n_nan_rub > 0:
    med_rub = df.loc[pretec21_mask, 'admission.rubric'].median()
    df['admission.rubric'].fillna(med_rub, inplace=True)
    print(f"admission.rubric: {n_nan_rub:,} NaN residuales imputados con mediana={med_rub:.2f}")
else:
    print(f"admission.rubric: sin NaN residuales ✓")

print(f"  min={df['admission.rubric'].min():.1f}  max={df['admission.rubric'].max():.1f}  "
      f"mean={df['admission.rubric'].mean():.1f}")

admission.rubric: sin NaN residuales ✓
  min=0.0  max=50.0  mean=34.4


## 5. MAR — Variables de actividades extracurriculares

AD14–AD17 tenían 3 columnas (physical.education, cultural.diffusion, student.society).  
AD18–AD20 cambiaron a 5 columnas LiFE (athletic.sports, art.culture, student.society.leadership,
life.work.mentoring, wellness.activities) con total.life.activities como contador.

La ausencia del dato indica **no participación**, no falta de registro — patrón MAR.

In [11]:
def parse_activity(val):
    """1 si el alumno participó (valor > 0 y no es 'Does not apply'), 0 en otro caso."""
    if pd.isna(val): return 0
    s = str(val).strip()
    if s in ('Does not apply', 'No information', '', '0', '0.0'): return 0
    try:   return int(float(s) > 0)
    except Exception: return 0

# Variable unificada: cualquier actividad universitaria
ACTIVITY_COLS = [
    'physical.education', 'cultural.diffusion', 'student.society',
    'total.life.activities', 'athletic.sports', 'art.culture',
    'student.society.leadership', 'life.work.mentoring', 'wellness.activities'
]
act_present = [c for c in ACTIVITY_COLS if c in df.columns]

# has_extracurriculars: participó en ALGUNA actividad (flag global)
# Usamos el indicador ya creado en el pipeline si existe, sino lo calculamos
if 'has_life_activities' in df.columns:
    df['has_extracurriculars'] = df['has_life_activities'].astype(int)
else:
    df['has_extracurriculars'] = df[act_present].apply(
        lambda row: int(any(parse_activity(v) for v in row)), axis=1)

# Variables granulares por tipo de actividad
physical_cols  = [c for c in ['physical.education','athletic.sports','wellness.activities'] if c in df.columns]
cultural_cols  = [c for c in ['cultural.diffusion','art.culture'] if c in df.columns]
social_cols    = [c for c in ['student.society','student.society.leadership','life.work.mentoring'] if c in df.columns]

df['has_physical'] = (df[physical_cols].apply(lambda r: any(parse_activity(v) for v in r), axis=1).astype(int)
                      if physical_cols else 0)
df['has_cultural'] = (df[cultural_cols].apply(lambda r: any(parse_activity(v) for v in r), axis=1).astype(int)
                      if cultural_cols else 0)
df['has_social']   = (df[social_cols].apply(lambda r: any(parse_activity(v) for v in r), axis=1).astype(int)
                      if social_cols else 0)

df.drop(columns=[c for c in act_present if c in df.columns], inplace=True)
df.drop(columns=[c for c in ['has_life_activities'] if c in df.columns], inplace=True)

print("Actividades extracurriculares:")
print(f"  has_extracurriculars : {df['has_extracurriculars'].mean()*100:.1f}% de alumnos con actividades")
print(f"  has_physical         : {df['has_physical'].mean()*100:.1f}%")
print(f"  has_cultural         : {df['has_cultural'].mean()*100:.1f}%")
print(f"  has_social           : {df['has_social'].mean()*100:.1f}%")

Actividades extracurriculares:
  has_extracurriculars : 46.0% de alumnos con actividades
  has_physical         : 80.7%
  has_cultural         : 38.8%
  has_social           : 43.1%


## 6. MNAR — `first.generation`

El dato falta de forma no aleatoria: tiende a faltar en perfiles socioeconómicos específicos.
La categoría -1 permite al modelo aprender el patrón de ausencia.

In [12]:
def encode_first_gen(val):
    """Yes=1, No=0, No information / Does not apply = -1 (MNAR: ausencia informativa)."""
    s = str(val).strip() if pd.notna(val) else ''
    if s == 'Yes': return  1
    if s == 'No':  return  0
    return -1  # 'No information', 'Does not apply', NaN

df['first_gen_enc'] = df['first.generation'].apply(encode_first_gen)
df.drop(columns=['first.generation'], inplace=True)

vals = df['first_gen_enc'].value_counts().sort_index()
print("first_gen_enc distribución:")
print(f"  -1 (MNAR/sin info): {vals.get(-1,0):>7,}  ({vals.get(-1,0)/len(df)*100:.1f}%)")
print(f"   0 (No):            {vals.get( 0,0):>7,}  ({vals.get( 0,0)/len(df)*100:.1f}%)")
print(f"   1 (Yes):           {vals.get( 1,0):>7,}  ({vals.get( 1,0)/len(df)*100:.1f}%)")

first_gen_enc distribución:
  -1 (MNAR/sin info):       0  (0.0%)
   0 (No):             71,744  (92.6%)
   1 (Yes):             5,773  (7.4%)


## 7. `max.degree.parents` → ordinal

In [13]:
EDU_ORD = {
    'No information': -1,
    'No degree':       0,
    'Undergraduate degree': 1,
    'Master degree':   2,
    'PhD':             3,
}

df['educ_padres_max'] = df['max.degree.parents'].map(EDU_ORD)
# NaN residuales (no mapeados) → -1 (sin información)
df['educ_padres_max'].fillna(-1, inplace=True)
df['educ_padres_max'] = df['educ_padres_max'].astype(int)
df.drop(columns=['max.degree.parents'], inplace=True)

print("educ_padres_max distribución:")
for k, v in sorted(df['educ_padres_max'].value_counts().items()):
    lbl = {-1:'sin info', 0:'sin título', 1:'licenciatura', 2:'maestría', 3:'doctorado'}
    print(f"  {k} ({lbl.get(k,'?')}): {v:>6,}")

educ_padres_max distribución:
  0 (sin título):  5,797
  1 (licenciatura): 42,509
  2 (maestría): 24,884
  3 (doctorado):  4,327


## 8. `parents.exatec` → binaria

In [14]:
df['parents_exatec_enc'] = df['parents.exatec'].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)
df.drop(columns=['parents.exatec'], inplace=True)
print(f"parents_exatec_enc: {df['parents_exatec_enc'].sum():,} alumnos con papás exatec "
      f"({df['parents_exatec_enc'].mean()*100:.1f}%)")

parents_exatec_enc: 10,855 alumnos con papás exatec (14.0%)


## 9. `gender` → `is_male`

In [15]:
df['is_male'] = df['gender'].map({'Male': 1, 'Female': 0}).fillna(0).astype(int)
df.drop(columns=['gender'], inplace=True)
print(f"is_male: {df['is_male'].mean()*100:.1f}% hombres")

is_male: 55.2% hombres


## 10. `tec.no.tec` → `estuvo_prepa_tec`

In [16]:
df['estuvo_prepa_tec'] = df['tec.no.tec'].map({'TEC': 1, 'NO TEC': 0}).fillna(0).astype(int)
df.drop(columns=['tec.no.tec'], inplace=True)
print(f"estuvo_prepa_tec: {df['estuvo_prepa_tec'].mean()*100:.1f}% con prepa TEC")

estuvo_prepa_tec: 47.7% con prepa TEC


## 11. `foreign` → one-hot (2 columnas)

In [17]:
# drop_first=True sobre 3 categorías (Local, Yes: National, Yes: Foreigner)
# → genera 'foreign_Yes: Foreigner' y 'foreign_Yes: National'
foreign_dummies = pd.get_dummies(df['foreign'], prefix='foreign', drop_first=True, dtype=int)
df = pd.concat([df, foreign_dummies], axis=1)
df.drop(columns=['foreign'], inplace=True)

for col in foreign_dummies.columns:
    print(f"  {col}: {df[col].sum():,} alumnos ({df[col].mean()*100:.1f}%)")

  foreign_Yes: Foreigner: 2,638 alumnos (3.4%)
  foreign_Yes: National: 16,979 alumnos (21.9%)


## 12. `zone.type` → ordinal + one-hot

In [18]:
# Ordinal: Rural=1, Semiurban=2, Urban=3, sin información=0
zone_ord = {'No information': 0, 'MISSING': 0, 'Rural': 1, 'Semiurban': 2, 'Urban': 3}
df['zone_enc'] = df['zone.type'].map(zone_ord).fillna(0).astype(int)

# One-hot: 'No information' → NaN para que get_dummies lo ignore
df['zone.type'] = df['zone.type'].replace({'No information': None, 'MISSING': None})
zone_dummies = pd.get_dummies(df['zone.type'], prefix='zone', drop_first=False, dtype=int)
df = pd.concat([df, zone_dummies], axis=1)
df.drop(columns=['zone.type'], inplace=True)

print("zone_enc distribución:")
for k, v in sorted(df['zone_enc'].value_counts().items()):
    lbl = {0:'sin info',1:'Rural',2:'Semiurbana',3:'Urbana'}
    print(f"  {k} ({lbl.get(k,'?')}): {v:>6,}")

zone_enc distribución:
  0 (sin info): 54,718
  1 (Rural):  1,355
  2 (Semiurbana):    788
  3 (Urbana): 20,656


## 13. `socioeconomic.level` → ordinal 0-7

In [19]:
sec_map = {'No information': 0, 'MISSING': 0,
           'Level 1': 1, 'Level 2': 2, 'Level 3': 3, 'Level 4': 4,
           'Level 5': 5, 'Level 6': 6, 'Level 7': 7}
df['socioec_enc'] = df['socioeconomic.level'].map(sec_map).fillna(0).astype(int)
df.drop(columns=['socioeconomic.level'], inplace=True)
print("socioec_enc:", df['socioec_enc'].value_counts().sort_index().to_dict())

socioec_enc: {0: 66446, 1: 38, 2: 27, 3: 252, 4: 1220, 5: 99, 6: 3301, 7: 6134}


## 14. `social.lag` → ordinal 0-3

In [20]:
lag_map = {'No information': 0, 'MISSING': 0, 'Low': 1, 'Medium': 2, 'High': 3}
df['social_lag_enc'] = df['social.lag'].map(lag_map).fillna(0).astype(int)
df.drop(columns=['social.lag'], inplace=True)
print("social_lag_enc:", df['social_lag_enc'].value_counts().sort_index().to_dict())

social_lag_enc: {0: 63984, 1: 12767, 2: 711, 3: 55}


## 15. `school` → label encode

In [21]:
le_school = LabelEncoder()
df['school_enc'] = le_school.fit_transform(df['school'].fillna('Unknown').astype(str))
print(f"school_enc: {df['school_enc'].nunique()} categorías")
print("  Mapeo:", dict(zip(le_school.classes_, le_school.transform(le_school.classes_))))
df.drop(columns=['school'], inplace=True)

school_enc: 6 categorías
  Mapeo: {'EAAD-Engineering and Sciences': np.int64(0), 'ECSG': np.int64(1), 'EHE-EAAD': np.int64(2), 'EIC': np.int64(3), 'EMCS': np.int64(4), 'EN': np.int64(5)}


## 16. `region` → label encode

In [22]:
le_region = LabelEncoder()
df['region_enc'] = le_region.fit_transform(df['region'].fillna('Unknown').astype(str))
print(f"region_enc: {df['region_enc'].nunique()} categorías")
print("  Mapeo:", dict(zip(le_region.classes_, le_region.transform(le_region.classes_))))
df.drop(columns=['region'], inplace=True)

region_enc: 5 categorías
  Mapeo: {'DR': np.int64(0), 'RCM': np.int64(1), 'RCS': np.int64(2), 'RM': np.int64(3), 'RO': np.int64(4)}


## 17. Renombrar `total.scholarship.loan` → `apoyo_financiero`

In [23]:
df.rename(columns={'total.scholarship.loan': 'apoyo_financiero'}, inplace=True)
print(f"apoyo_financiero: min={df['apoyo_financiero'].min():.2f}  "
      f"max={df['apoyo_financiero'].max():.2f}  "
      f"mean={df['apoyo_financiero'].mean():.3f}")
print(f"  Alumnos sin apoyo (=0): {(df['apoyo_financiero']==0).mean()*100:.1f}%")

apoyo_financiero: min=0.00  max=1.00  mean=0.236
  Alumnos sin apoyo (=0): 51.1%


## 18. Validación del dataset final

In [24]:
print("=" * 60)
print("VALIDACIÓN FINAL — dataset_engineered.csv")
print("=" * 60)
print(f"Shape          : {df.shape}")
print(f"Columnas       : {df.shape[1]}")
print(f"Registros      : {df.shape[0]:,}")
print(f"Tasa deserción : {(df['retention']==0).mean()*100:.2f}%")

# NaN residuales
nulls = df.isnull().sum()
null_df = nulls[nulls > 0]
if len(null_df) == 0:
    print("\n✓ Sin NaN residuales en ninguna columna")
else:
    print(f"\n⚠ Columnas con NaN residuales:")
    for col, n in null_df.items():
        print(f"  {col}: {n:,} ({n/len(df)*100:.1f}%)")

print(f"\nColumnas del dataset final:")
for col in df.columns:
    dtype = df[col].dtype
    n_uniq = df[col].nunique()
    print(f"  {col:<40} {str(dtype):<10} {n_uniq:>5} valores únicos")

VALIDACIÓN FINAL — dataset_engineered.csv
Shape          : (77517, 37)
Columnas       : 37
Registros      : 77,517
Tasa deserción : 8.79%

✓ Sin NaN residuales en ninguna columna

Columnas del dataset final:
  generation                               str            7 valores únicos
  age                                      int64         29 valores únicos
  PNA                                      float64     2846 valores únicos
  online.test                              int64          2 valores únicos
  english.evaluation                       int64          8 valores únicos
  admission.rubric                         float64    10898 valores únicos
  general.math.eval                        float64     2753 valores únicos
  retention                                int64          2 valores únicos
  FTE                                      float64       60 valores únicos
  apoyo_financiero                         float64     2845 valores únicos
  regime                                  

In [25]:
# Distribución de retention por régimen
print("\nTasa de deserción por régimen:")
for reg in ['PreTec21', 'Tec21']:
    sub = df[df['regime'] == reg]
    rate = (sub['retention'] == 0).mean() * 100
    print(f"  {reg}: {len(sub):,} alumnos  →  {rate:.2f}% deserción")

# Variables de missingness que venían del pipeline
miss_cols = [c for c in df.columns if c.startswith('has_') or
             c in ['took_admission_test','first_gen_present','parents_edu_present',
                   'has_socioeconomic_data','has_social_lag_data','has_zone_data']]
print(f"\nIndicadores de missingness conservados: {miss_cols}")


Tasa de deserción por régimen:
  PreTec21: 53,010 alumnos  →  8.84% deserción
  Tec21: 24,507 alumnos  →  8.68% deserción

Indicadores de missingness conservados: ['took_admission_test', 'first_gen_present', 'parents_edu_present', 'has_socioeconomic_data', 'has_social_lag_data', 'has_zone_data', 'has_extracurriculars', 'has_physical', 'has_cultural', 'has_social']


## 19. Exportar `dataset_engineered.csv`

In [26]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"✓ Dataset exportado: {OUTPUT_CSV}")
print(f"  {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\nEjemplo de primeras filas:")
print(df.head(3).to_string())

✓ Dataset exportado: ../dataset_engineered.csv
  77,517 filas × 37 columnas

Ejemplo de primeras filas:
  generation  age    PNA  online.test  english.evaluation  admission.rubric  general.math.eval  retention   FTE  apoyo_financiero    regime  took_admission_test  first_gen_present  parents_edu_present  has_socioeconomic_data  has_social_lag_data  has_zone_data  admission_test_norm  has_extracurriculars  has_physical  has_cultural  has_social  first_gen_enc  educ_padres_max  parents_exatec_enc  is_male  estuvo_prepa_tec  foreign_Yes: Foreigner  foreign_Yes: National  zone_enc  zone_Rural  zone_Semiurban  zone_Urban  socioec_enc  social_lag_enc  school_enc  region_enc
0       AD14   19  87.00            0                   6         33.920290               75.5          1  1.08          0.000000  PreTec21                    0                  0                    0                       0                    0              0             0.764667                     0             0      

---

## Resumen del pipeline de preprocesamiento

```
dataset.csv (raw)
    ↓ 01_missingness_diagnosis.ipynb
    Diagnóstico de tipos de missingness (MCAR / MAR / MNAR)

    ↓ 02_imputation_pipeline.ipynb
    dataset_imputed.csv
    - Imputación KNN de numéricos (admission.test, general.math.eval)
    - Imputación MICE de admission.rubric
    - Imputación RF de first.generation y max.degree.parents
    - Indicadores binarios de missingness original

    ↓ 03_feature_engineering.ipynb  ← ESTE NOTEBOOK
    dataset_engineered.csv
    - Normalización de escalas (admission_test_norm)
    - Encodings de categóricas (ordinal, binario, one-hot)
    - Variables compuestas de actividades (has_extracurriculars, has_physical, etc.)
    - MNAR encoding para first_gen_enc (-1/0/1)
    - Renombres semánticos (apoyo_financiero)
    - Eliminación de columnas con fuga de datos

    ↓ 02_Experimentacion_Unificada.ipynb
    - K-Means por régimen
    - Logistic Regression / Decision Tree / Random Forest / XGBoost
    - Análisis SHAP e invarianza PreTec21 ↔ Tec21
```